# VideoAI GPU worker (free Google Colab GPU)

This notebook gives the **VideoAI** app a free NVIDIA GPU so it can generate realistic talking-avatar videos:

* **EchoMimicV2** – half body with **hand gestures** from one photo + voice (closest to HeyGen). Apache-2.0.
* **SadTalker** – realistic face / lip-sync. Apache-2.0. Used automatically if EchoMimicV2 fails (e.g. out of GPU memory).

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all** (first run downloads ~15 GB and takes roughly 20–30 minutes).
3. The last cell prints a **URL** and a **TOKEN**. In VideoAI open **System check → GPU worker**, paste both, press *Connect*.
4. In the editor choose **Look & feel → Animation engine → HeyGen-like (EchoMimicV2, Colab GPU)** and render.
5. **Keep this tab open** while rendering. Free Colab disconnects after inactivity / a few hours; then just *Run all* again and paste the new URL.

Rendering on the free T4 GPU is slow (EchoMimicV2: very roughly 10–30 minutes per 10 seconds of video; SadTalker ~1 minute).

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
#@title 2. Options
INSTALL_ECHOMIMIC = True  #@param {type:"boolean"}
INSTALL_SADTALKER = True  #@param {type:"boolean"}

In [ ]:
#@title 3. Base tools (ffmpeg, uv, Python 3.10 for the models)
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg git-lfs > /dev/null
!pip -q install uv fastapi "uvicorn[standard]" python-multipart huggingface_hub
!uv python install 3.10

In [ ]:
#@title 4. Install SadTalker (face / lip-sync)
import subprocess
if INSTALL_SADTALKER:
    script = r'''
set -e
cd /content
[ -d SadTalker ] || git clone --depth 1 https://github.com/OpenTalker/SadTalker.git
[ -x env_sadtalker/bin/python ] || uv venv --python 3.10 env_sadtalker
P=/content/env_sadtalker/bin/python
uv pip install --python $P "setuptools<70" wheel
uv pip install --python $P torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $P numpy==1.23.4 face_alignment==1.3.5 imageio==2.19.3 imageio-ffmpeg==0.4.7 librosa==0.9.2 numba==0.58.1 resampy==0.3.1 pydub==0.25.1 scipy==1.10.1 kornia==0.6.8 tqdm yacs==0.1.8 pyyaml joblib==1.1.0 scikit-image==0.19.3 opencv-python==4.7.0.72 basicsr==1.4.2 facexlib==0.3.0 gfpgan==1.3.8 av safetensors
cd SadTalker
mkdir -p checkpoints gfpgan/weights
R=https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc
for f in mapping_00109-model.pth.tar mapping_00229-model.pth.tar SadTalker_V0.0.2_256.safetensors SadTalker_V0.0.2_512.safetensors; do
  [ -s checkpoints/$f ] || wget -q -O checkpoints/$f $R/$f
done
[ -s gfpgan/weights/alignment_WFLW_4HG.pth ] || wget -q -O gfpgan/weights/alignment_WFLW_4HG.pth https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth
[ -s gfpgan/weights/detection_Resnet50_Final.pth ] || wget -q -O gfpgan/weights/detection_Resnet50_Final.pth https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth
[ -s gfpgan/weights/GFPGANv1.4.pth ] || wget -q -O gfpgan/weights/GFPGANv1.4.pth https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
[ -s gfpgan/weights/parsing_parsenet.pth ] || wget -q -O gfpgan/weights/parsing_parsenet.pth https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth
echo SADTALKER_OK
'''
    r = subprocess.run(["bash", "-c", script], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    print("SadTalker installed" if "SADTALKER_OK" in r.stdout else "SadTalker install FAILED (see the messages above)")

In [ ]:
#@title 5. Install EchoMimicV2 (half body + hand gestures)
import subprocess
if INSTALL_ECHOMIMIC:
    script = r'''
set -e
cd /content
[ -d echomimic_v2 ] || git clone --depth 1 https://github.com/antgroup/echomimic_v2.git
[ -x env_echomimic/bin/python ] || uv venv --python 3.10 env_echomimic
P=/content/env_echomimic/bin/python
uv pip install --python $P torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python $P -r echomimic_v2/requirements.txt
uv pip install --python $P --no-deps facenet_pytorch==2.6.0
cd echomimic_v2
python - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download("BadToBest/EchoMimicV2", local_dir="pretrained_weights")
snapshot_download("stabilityai/sd-vae-ft-mse", local_dir="pretrained_weights/sd-vae-ft-mse")
snapshot_download("lambdalabs/sd-image-variations-diffusers", local_dir="pretrained_weights/sd-image-variations-diffusers",
                  allow_patterns=["unet/*", "image_encoder/*", "feature_extractor/*", "scheduler/*", "model_index.json"])
PY
mkdir -p pretrained_weights/audio_processor
[ -s pretrained_weights/audio_processor/tiny.pt ] || wget -q -O pretrained_weights/audio_processor/tiny.pt https://openaipublic.azureedge.net/main/whisper/models/65147644a518d12f04e32d6f3b26facc3f8dd46e5390956a9424a650c0ce22b9/tiny.pt
echo ECHOMIMIC_OK
'''
    r = subprocess.run(["bash", "-c", script], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    print("EchoMimicV2 installed" if "ECHOMIMIC_OK" in r.stdout else "EchoMimicV2 install FAILED - the worker will use SadTalker instead")

In [ ]:
%%writefile /content/gpu_worker.py
"""VideoAI GPU worker: runs neural talking-avatar engines on a GPU (Google Colab) for the VideoAI app.

Protocol (all requests need header X-Token):
    GET  /health             engines + GPU info
    POST /jobs               multipart: engine=echomimic|sadtalker, fps, image (PNG), audio (WAV)  -> {"id"}
    GET  /jobs/{id}          {"status": queued|running|done|failed, "message", "engine_used"}
    GET  /jobs/{id}/video    the MP4 (same size as the input image, with the input audio)

Engines (installed by the notebook, each in its own Python 3.10 virtualenv):
    echomimic -> EchoMimicV2 (https://github.com/antgroup/echomimic_v2, Apache-2.0): half body + hand gestures
    sadtalker -> SadTalker  (https://github.com/OpenTalker/SadTalker, Apache-2.0): face / head
If EchoMimicV2 fails (e.g. out of GPU memory) the job automatically falls back to SadTalker and says so.

Environment: WORKER_TOKEN, WORK_DIR, ECHOMIMIC_DIR, ECHOMIMIC_PY, SADTALKER_DIR, SADTALKER_PY,
             WORKER_STUB=1 (testing only: "engines" just turn the image + audio into a video with ffmpeg).
"""
import os
import queue
import re
import secrets
import shutil
import subprocess
import threading
import time
import uuid
import wave
from pathlib import Path

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse

TOKEN = os.environ.get("WORKER_TOKEN") or secrets.token_urlsafe(16)
WORK = Path(os.environ.get("WORK_DIR", "/content/videoai_jobs"))
WORK.mkdir(parents=True, exist_ok=True)
STUB = os.environ.get("WORKER_STUB") == "1"
ENG = {
    "echomimic": (Path(os.environ.get("ECHOMIMIC_DIR", "/content/echomimic_v2")), os.environ.get("ECHOMIMIC_PY", "/content/env_echomimic/bin/python")),
    "sadtalker": (Path(os.environ.get("SADTALKER_DIR", "/content/SadTalker")), os.environ.get("SADTALKER_PY", "/content/env_sadtalker/bin/python")),
}
app = FastAPI(title="VideoAI GPU worker")
jobs: dict[str, dict] = {}
q: "queue.Queue[str]" = queue.Queue()


class EngineError(RuntimeError):
    pass


def _auth(token: str | None) -> None:
    if not token or not secrets.compare_digest(token, TOKEN):
        raise HTTPException(401, "bad token")


def _run(cmd: list[str], cwd: Path | None = None, timeout: int = 4 * 3600) -> subprocess.CompletedProcess:
    r = subprocess.run(cmd, cwd=str(cwd) if cwd else None, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        tail = (r.stderr or r.stdout or "").strip().splitlines()[-8:]
        raise EngineError(" | ".join(tail)[-700:] or f"exit code {r.returncode}")
    return r


def _ffmpeg(*args: str) -> None:
    _run(["ffmpeg", "-y", "-loglevel", "error", *args])


def _image_size(path: Path) -> tuple[int, int]:
    out = _run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height", "-of", "csv=p=0", str(path)]).stdout
    w, h = out.strip().split(",")[:2]
    return int(w), int(h)


def _audio_seconds(path: Path) -> float:
    with wave.open(str(path)) as w:
        return w.getnframes() / w.getframerate()


def _finish(video: Path, image: Path, audio: Path, out: Path, fps: int, crop: str | None = None) -> Path:
    """Normalise any engine output: optional crop, scale to the input image size, given fps, the job's own audio."""
    w, h = _image_size(image)
    w, h = w - w % 2, h - h % 2
    vf = (crop + "," if crop else "") + f"scale={w}:{h},fps={fps}"
    _ffmpeg("-i", str(video), "-i", str(audio), "-map", "0:v:0", "-map", "1:a:0", "-vf", vf, "-c:v", "libx264",
            "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest", str(out))
    return out


def engine_status(name: str) -> dict:
    if STUB:
        return {"ready": True, "stub": True}
    d, py = ENG[name]
    problems = []
    if not Path(py).exists():
        problems.append(f"python env missing ({py})")
    if name == "sadtalker":
        if not (d / "inference.py").exists():
            problems.append("code missing")
        if not (d / "checkpoints" / "SadTalker_V0.0.2_256.safetensors").exists():
            problems.append("weights missing")
    else:
        if not any((d / s).exists() for s in ("infer_acc.py", "infer.py")):
            problems.append("code missing")
        if not (d / "pretrained_weights" / "denoising_unet.pth").exists():
            problems.append("weights missing")
    return {"ready": not problems, "problems": problems}


# ----------------------------------------------------------------------------- engines
def run_stub(image: Path, audio: Path, job: Path) -> Path:
    raw = job / "stub.mp4"
    _ffmpeg("-loop", "1", "-i", str(image), "-i", str(audio), "-shortest", "-r", "25", "-pix_fmt", "yuv420p",
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", str(raw))
    return raw


def run_sadtalker(image: Path, audio: Path, job: Path) -> Path:
    d, py = ENG["sadtalker"]
    out = job / "sadtalker"
    _run([py, "inference.py", "--driven_audio", str(audio), "--source_image", str(image), "--result_dir", str(out),
          "--still", "--preprocess", "full", "--enhancer", "gfpgan"], cwd=d)
    vids = sorted(out.glob("*.mp4"), key=lambda p: p.stat().st_mtime) or sorted(out.rglob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if not vids:
        raise EngineError("SadTalker produced no video")
    return vids[-1]


ECHO_SIZE = 768


def run_echomimic(image: Path, audio: Path, job: Path, fps: int) -> tuple[Path, str]:
    """EchoMimicV2 needs a square reference image and a pose sequence (it provides demo poses, which drive the hands)."""
    d, py = ENG["echomimic"]
    script = "infer_acc.py" if (d / "infer_acc.py").exists() else "infer.py"
    config = d / "configs" / "prompts" / ("infer_acc.yaml" if script == "infer_acc.py" else "infer.yaml")
    helptext = _run([py, script, "--help"], cwd=d, timeout=600).stdout
    has = lambda flag: re.search(rf"(^|\s){re.escape(flag)}[\s,=]", helptext) is not None
    needed = ["--ref_images_dir", "--refimg_name", "--audio_dir", "--audio_name", "--pose_dir", "--pose_name"]
    missing = [f for f in needed if not has(f)]
    if missing:
        raise EngineError(f"EchoMimicV2's {script} has a different interface than expected (missing {', '.join(missing)})")
    # square, edge-extended reference so the person keeps their framing; cropped back afterwards
    w, h = _image_size(image)
    side = max(w, h)
    ref_dir, aud_dir = job / "ref", job / "aud"
    ref_dir.mkdir(exist_ok=True)
    aud_dir.mkdir(exist_ok=True)
    pad_x, pad_y = (side - w) // 2, (side - h) // 2
    _ffmpeg("-i", str(image), "-vf", f"pad={side}:{side}:{pad_x}:{pad_y}:color=0x808080,scale={ECHO_SIZE}:{ECHO_SIZE}", str(ref_dir / "ref.png"))
    _ffmpeg("-i", str(audio), "-ar", "16000", "-ac", "1", str(aud_dir / "a.wav"))
    pose_root = d / "assets" / "halfbody_demo" / "pose"
    poses = sorted(p for p in pose_root.iterdir() if p.is_dir() and any(p.glob("*.npy"))) if pose_root.exists() else []
    if not poses:
        raise EngineError(f"no pose sequences found in {pose_root}")
    pose = poses[int(time.time()) % len(poses)]  # vary gestures between clips
    n_pose = len(list(pose.glob("*.npy")))
    n_need = int(_audio_seconds(audio) * 24) + 1
    if n_need > n_pose:
        raise EngineError(f"clip is {n_need / 24:.1f}s but the pose sequence only has {n_pose / 24:.1f}s; split the scene")
    args = [py, script]
    if has("--config"):
        args += ["--config", str(config)]
    for flag, val in (("-W", ECHO_SIZE), ("-H", ECHO_SIZE), ("-L", n_need), ("--fps", 24)):
        if has(flag):
            args += [flag, str(val)]
    args += ["--ref_images_dir", str(ref_dir), "--refimg_name", "ref.png", "--audio_dir", str(aud_dir), "--audio_name", "a.wav",
             "--pose_dir", str(pose_root), "--pose_name", pose.name]
    t0 = time.time()
    _run(args, cwd=d)
    vids = [p for p in (d / "output").rglob("*.mp4") if p.stat().st_mtime >= t0 - 1]
    vids.sort(key=lambda p: ("sig" in p.name, p.stat().st_mtime))
    if not vids:
        raise EngineError("EchoMimicV2 produced no video")
    # undo the square padding: crop the original aspect back out of the square output
    crop = f"crop=iw*{w}/{side}:ih*{h}/{side}:iw*{pad_x}/{side}:ih*{pad_y}/{side}"
    return vids[-1], crop


def process(job_id: str) -> None:
    j = jobs[job_id]
    jd = WORK / job_id
    j["status"] = "running"
    t0 = time.time()
    try:
        image, audio, out = jd / "in.png", jd / "in.wav", jd / "out.mp4"
        if STUB:
            _finish(run_stub(image, audio, jd), image, audio, out, j["fps"])
            j["engine_used"], j["message"] = j["engine"], f"stub {j['engine']}"
        elif j["engine"] == "echomimic":
            try:
                raw, crop = run_echomimic(image, audio, jd, j["fps"])
                _finish(raw, image, audio, out, j["fps"], crop)
                j["engine_used"], j["message"] = "echomimic", f"EchoMimicV2 on the GPU in {time.time() - t0:.0f}s"
            except Exception as e:  # out of memory, interface change, missing weights ...
                raw = run_sadtalker(image, audio, jd)
                _finish(raw, image, audio, out, j["fps"])
                j["engine_used"] = "sadtalker"
                j["message"] = f"EchoMimicV2 failed ({str(e)[:300]}); used SadTalker on the GPU instead"
        else:
            _finish(run_sadtalker(image, audio, jd), image, audio, out, j["fps"])
            j["engine_used"], j["message"] = "sadtalker", f"SadTalker on the GPU in {time.time() - t0:.0f}s"
        j["status"] = "done"
    except Exception as e:
        j["status"], j["message"] = "failed", str(e)[:800]
    j["seconds"] = round(time.time() - t0, 1)


def _loop() -> None:
    while True:
        process(q.get())


threading.Thread(target=_loop, daemon=True).start()


@app.get("/health")
def health(x_token: str | None = Header(default=None)):
    _auth(x_token)
    gpu = ""
    if shutil.which("nvidia-smi"):
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return {"ok": True, "gpu": gpu or ("stub" if STUB else "none"), "queued": q.qsize(),
            "engines": {k: engine_status(k) for k in ENG}}


@app.post("/jobs")
async def create_job(engine: str = Form(...), fps: int = Form(24), image: UploadFile = File(...), audio: UploadFile = File(...),
                     x_token: str | None = Header(default=None)):
    _auth(x_token)
    if engine not in ENG:
        raise HTTPException(422, "engine must be echomimic or sadtalker")
    st = engine_status(engine)
    if not st["ready"] and not (engine == "echomimic" and engine_status("sadtalker")["ready"]):
        raise HTTPException(503, f"{engine} is not installed: {st.get('problems')}")
    job_id = uuid.uuid4().hex[:12]
    jd = WORK / job_id
    jd.mkdir(parents=True)
    (jd / "in.png").write_bytes(await image.read())
    (jd / "in.wav").write_bytes(await audio.read())
    jobs[job_id] = {"status": "queued", "engine": engine, "fps": max(1, min(60, fps)), "message": "", "engine_used": None}
    q.put(job_id)
    return {"id": job_id}


@app.get("/jobs/{job_id}")
def job_status(job_id: str, x_token: str | None = Header(default=None)):
    _auth(x_token)
    if job_id not in jobs:
        raise HTTPException(404, "unknown job")
    return jobs[job_id]


@app.get("/jobs/{job_id}/video")
def job_video(job_id: str, x_token: str | None = Header(default=None)):
    _auth(x_token)
    out = WORK / job_id / "out.mp4"
    if jobs.get(job_id, {}).get("status") != "done" or not out.exists():
        raise HTTPException(404, "not ready")
    return FileResponse(out, media_type="video/mp4")

In [ ]:
#@title 6. Self-test both engines on their own example files (shows real speed on this GPU)
import glob, os, subprocess, sys, time, wave
sys.path.insert(0, "/content")
os.environ["WORKER_STUB"] = "0"
import importlib, gpu_worker
importlib.reload(gpu_worker)
from pathlib import Path

def short_wav(src, dst, seconds=3):
    r = wave.open(src); w = wave.open(dst, "wb"); w.setparams(r.getparams())
    w.writeframes(r.readframes(int(r.getframerate() * seconds))); w.close()

job = Path("/content/selftest"); job.mkdir(exist_ok=True)
for name in ("sadtalker", "echomimic"):
    st = gpu_worker.engine_status(name)
    if not st["ready"]:
        print(f"{name}: not installed {st['problems']}"); continue
    d = gpu_worker.ENG[name][0]
    imgs = sorted(glob.glob(str(d / ("examples/source_image/*.png" if name == "sadtalker" else "assets/halfbody_demo/refimag/**/*.png")), recursive=True))
    wavs = sorted(glob.glob(str(d / ("examples/driven_audio/*.wav" if name == "sadtalker" else "assets/halfbody_demo/audio/**/*.wav")), recursive=True))
    if not imgs or not wavs:
        print(f"{name}: no example files found"); continue
    short_wav(wavs[0], str(job / f"{name}.wav"))
    t = time.time()
    try:
        if name == "sadtalker":
            out = gpu_worker.run_sadtalker(Path(imgs[0]), job / f"{name}.wav", job)
        else:
            out, _ = gpu_worker.run_echomimic(Path(imgs[0]), job / f"{name}.wav", job, 24)
        print(f"{name}: OK, 3 s of video in {time.time() - t:.0f} s -> {out}")
    except Exception as e:
        print(f"{name}: FAILED after {time.time() - t:.0f} s: {e}")

In [ ]:
#@title 7. Start the worker and get the URL + TOKEN for VideoAI (keep this tab open)
import os, re, secrets, subprocess, threading, time
TOKEN = secrets.token_urlsafe(12)
env = {**os.environ, "WORKER_TOKEN": TOKEN, "WORKER_STUB": "0"}
server = subprocess.Popen(["python", "-m", "uvicorn", "gpu_worker:app", "--host", "127.0.0.1", "--port", "8765"], cwd="/content", env=env)
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("/content/cloudflared", 0o755)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
time.sleep(3)
print("=" * 70)
print("Paste these into VideoAI -> System check -> GPU worker:")
print("URL:  ", URL)
print("TOKEN:", TOKEN)
print("=" * 70)

In [ ]:
#@title 8. (Optional) keep the session busy while rendering - stop this cell when you are done
import time, json, urllib.request
while True:
    try:
        req = urllib.request.Request("http://127.0.0.1:8765/health", headers={"X-Token": TOKEN})
        h = json.loads(urllib.request.urlopen(req, timeout=10).read())
        print(time.strftime("%H:%M:%S"), "worker OK, jobs waiting:", h["queued"])
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "worker not responding:", e)
    time.sleep(60)